In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q diffusers transformers accelerate safetensors peft

In [ ]:
import diffusers
import transformers
import peft

print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

pipe = pipe.to("cuda")

print("Model loaded successfully!")

In [ ]:
prompt = "a beautiful red flower in a garden"

image = pipe(
    prompt,
    num_inference_steps=20,
    guidance_scale=7.5
).images[0]

display(image)

In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ClemenKok/digimon-blip-captions")

print(dataset)

In [ ]:
sample = dataset["train"][0]

print("Caption:", sample["text"])
print("Image:", sample["image"])

display(sample["image"])

In [ ]:
for i in range(5):
    sample = dataset["train"][i]
    print(f"\nSample {i + 1}")
    print("Caption:", sample["text"])
    print("Image mode:", sample["image"].mode)
    print("Image size:", sample["image"].size)
    display(sample["image"])

In [ ]:
from PIL import Image

sample_image = dataset["train"][0]["image"]

processed_image = sample_image.convert("RGB")
processed_image = processed_image.resize((64, 64))

print("Original:", sample_image.mode, sample_image.size)
print("Processed:", processed_image.mode, processed_image.size)

display(processed_image)

In [ ]:
from PIL import Image

def preprocess_image(image):
    image = image.convert("RGB")
    image = image.resize((64, 64))
    return image

processed_dataset = []

for sample in dataset["train"]:
    image = preprocess_image(sample["image"])
    caption = sample["text"]

    processed_dataset.append({
        "image": image,
        "text": caption
    })

print("Processed samples:", len(processed_dataset))
print("First caption:", processed_dataset[0]["text"])
print("First image:", processed_dataset[0]["image"].mode,
      processed_dataset[0]["image"].size)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

image_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

class DigimonDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx]["image"]
        caption = self.data[idx]["text"]

        image = image_transform(image)

        return {
            "image": image,
            "caption": caption
        }

train_dataset = DigimonDataset(processed_dataset)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

print("Dataset size:", len(train_dataset))
print("Number of batches:", len(train_loader))

In [ ]:
import os

os.makedirs("task1_custom_dataset", exist_ok=True)

for i, sample in enumerate(processed_dataset):
    image_path = f"task1_custom_dataset/image_{i:04d}.png"
    sample["image"].save(image_path)

with open("task1_custom_dataset/captions.txt", "w", encoding="utf-8") as f:
    for i, sample in enumerate(processed_dataset):
        f.write(f"image_{i:04d}.png|{sample['text']}\n")

print("Custom dataset saved successfully!")
print("Images:", len(processed_dataset))
print("Caption file: task1_custom_dataset/captions.txt")

In [ ]:
import torch

gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total GPU memory: {gpu_memory:.2f} GB")

In [ ]:
from PIL import Image
import os

os.makedirs("task1_training_dataset/images", exist_ok=True)

for i, sample in enumerate(processed_dataset):
    image = sample["image"].convert("RGB")
    image = image.resize((512, 512))

    image.save(
        f"task1_training_dataset/images/image_{i:04d}.png"
    )

with open("task1_training_dataset/captions.txt", "w", encoding="utf-8") as f:
    for i, sample in enumerate(processed_dataset):
        f.write(f"image_{i:04d}.png|{sample['text']}\n")

print("512x512 training dataset created!")
print("Images:", len(processed_dataset))

In [ ]:
sample_image = Image.open(
    "task1_training_dataset/images/image_0000.png"
)

with open("task1_training_dataset/captions.txt", "r", encoding="utf-8") as f:
    first_caption = f.readline().strip()

print("Image:", sample_image.size, sample_image.mode)
print("Caption:", first_caption)

display(sample_image)

In [ ]:
!git clone -q https://github.com/huggingface/diffusers.git

In [ ]:
!pip install -q -r diffusers/examples/text_to_image/requirements.txt

In [ ]:
from datasets import Dataset, Image as HFImage
import os

image_paths = []
captions = []

for i, sample in enumerate(processed_dataset):
    image_paths.append(
        f"task1_training_dataset/images/image_{i:04d}.png"
    )
    captions.append(sample["text"])

train_data = Dataset.from_dict({
    "image": image_paths,
    "text": captions
})

train_data = train_data.cast_column("image", HFImage())

print(train_data)
print("\nFirst sample:")
print(train_data[0])

In [ ]:
train_data.save_to_disk("task1_hf_dataset")

print("Dataset saved successfully!")

In [ ]:
import os

os.makedirs("task1_lora_output", exist_ok=True)

print("LoRA training configuration")
print("----------------------------")
print("Base model: Stable Diffusion 1.5")
print("Dataset: 1,071 Digimon image-caption pairs")
print("Resolution: 512x512")
print("Batch size: 1")
print("Gradient accumulation: 4")
print("Epochs: 1")
print("Learning rate: 1e-4")
print("LoRA rank: 4")
print("Mixed precision: fp16")
print("Output folder: task1_lora_output")

In [ ]:
!ls diffusers/examples/text_to_image/

In [ ]:
!accelerate launch diffusers/examples/text_to_image/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --dataset_name="task1_hf_dataset" \
  --image_column="image" \
  --caption_column="text" \
  --resolution=512 \
  --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --num_train_epochs=1 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --mixed_precision="fp16" \
  --rank=4 \
  --output_dir="task1_lora_output" \
  --checkpointing_steps=500 \
  --seed=42

In [ ]:
!pip install -q -e ./diffusers

In [ ]:
import diffusers

print("Diffusers version:", diffusers.__version__)

In [ ]:
from datasets import load_from_disk

train_data = load_from_disk("task1_hf_dataset")

print(train_data)
print("Number of samples:", len(train_data))
print("Columns:", train_data.column_names)

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!accelerate launch \
  --mixed_precision=fp16 \
  diffusers/examples/text_to_image/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --dataset_name="task1_hf_dataset" \
  --image_column="image" \
  --caption_column="text" \
  --resolution=512 \
  --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --num_train_epochs=1 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --mixed_precision="fp16" \
  --rank=4 \
  --output_dir="task1_lora_output" \
  --checkpointing_steps=500 \
  --seed=42

In [ ]:
!pip install -q --upgrade "torchao==0.18.0"

In [ ]:
import torch
import torchao
import diffusers

print("PyTorch:", torch.__version__)
print("TorchAO:", torchao.__version__)
print("Diffusers:", diffusers.__version__)
print("CUDA:", torch.cuda.is_available())

In [ ]:
!accelerate launch \
  --mixed_precision=fp16 \
  diffusers/examples/text_to_image/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --dataset_name="task1_hf_dataset" \
  --image_column="image" \
  --caption_column="text" \
  --resolution=512 \
  --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --num_train_epochs=1 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --mixed_precision="fp16" \
  --rank=4 \
  --output_dir="task1_lora_output" \
  --checkpointing_steps=500 \
  --seed=42

In [ ]:
!accelerate launch \
  --mixed_precision=fp16 \
  diffusers/examples/text_to_image/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="task1_training_dataset" \
  --image_column="image" \
  --caption_column="text" \
  --resolution=512 \
  --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --num_train_epochs=1 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --mixed_precision="fp16" \
  --rank=4 \
  --output_dir="task1_lora_output" \
  --checkpointing_steps=500 \
  --seed=42

In [ ]:
import json
import os

dataset_dir = "task1_training_dataset"
captions_file = os.path.join(dataset_dir, "captions.txt")
metadata_file = os.path.join(dataset_dir, "metadata.jsonl")

with open(captions_file, "r", encoding="utf-8") as f:
    captions = [line.strip() for line in f if line.strip()]

with open(metadata_file, "w", encoding="utf-8") as f:
    for i, caption in enumerate(captions):
        item = {
            "file_name": f"images/image_{i:04d}.png",
            "text": caption
        }
        f.write(json.dumps(item) + "\n")

print("Metadata created:", metadata_file)
print("Number of entries:", len(captions))

# Show first 2 entries
with open(metadata_file, "r", encoding="utf-8") as f:
    print(f.readline().strip())
    print(f.readline().strip())

In [ ]:
import json

metadata_file = "task1_training_dataset/metadata.jsonl"

cleaned_entries = []

with open(metadata_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        # Remove the filename prefix from the caption
        if "|" in item["text"]:
            item["text"] = item["text"].split("|", 1)[1]

        cleaned_entries.append(item)

with open(metadata_file, "w", encoding="utf-8") as f:
    for item in cleaned_entries:
        f.write(json.dumps(item) + "\n")

print("Metadata cleaned successfully.")
print("First entry:", cleaned_entries[0])
print("Second entry:", cleaned_entries[1])

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "imagefolder",
    data_dir="task1_training_dataset"
)

print(dataset)
print("Columns:", dataset["train"].column_names)
print("Number of samples:", len(dataset["train"]))

print("\nFirst sample:")
print(dataset["train"][0])

In [ ]:
!accelerate launch \
  --mixed_precision=fp16 \
  diffusers/examples/text_to_image/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --train_data_dir="task1_training_dataset" \
  --image_column="image" \
  --caption_column="text" \
  --resolution=512 \
  --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --num_train_epochs=1 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --mixed_precision="fp16" \
  --rank=4 \
  --output_dir="task1_lora_output" \
  --checkpointing_steps=500 \
  --seed=42

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to("cuda")

# Load your trained LoRA
pipe.load_lora_weights(
    "task1_lora_output",
    weight_name="pytorch_lora_weights.safetensors"
)

print("LoRA loaded successfully!")

In [ ]:
prompt = "a drawing of a cartoon character holding a wrench"

image = pipe(
    prompt,
    num_inference_steps=30,
    guidance_scale=7.5
).images[0]

display(image)

In [ ]:
pipe.unload_lora_weights()

print("LoRA unloaded successfully!")

In [ ]:
prompt = "a drawing of a cartoon character holding a wrench"

base_image = pipe(
    prompt,
    num_inference_steps=30,
    guidance_scale=7.5
).images[0]

display(base_image)

In [ ]:
base_image.save("task1_base_image.png")

print("Base image saved successfully!")

In [ ]:
pipe.load_lora_weights(
    "task1_lora_output",
    weight_name="pytorch_lora_weights.safetensors"
)

prompt = "a drawing of a cartoon character holding a wrench"

lora_image = pipe(
    prompt,
    num_inference_steps=30,
    guidance_scale=7.5
).images[0]

lora_image.save("task1_lora_image.png")

print("LoRA image saved successfully!")

In [ ]:
from PIL import Image, ImageDraw

base_image = Image.open("task1_base_image.png")
lora_image = Image.open("task1_lora_image.png")

comparison = Image.new(
    "RGB",
    (base_image.width * 2, base_image.height)
)

comparison.paste(base_image, (0, 0))
comparison.paste(lora_image, (base_image.width, 0))

display(comparison)

comparison.save("task1_base_vs_lora_comparison.png")

print("Comparison saved successfully!")

In [ ]:
from PIL import Image
import numpy as np

base = Image.open("task1_base_image.png").convert("RGB")
lora = Image.open("task1_lora_image.png").convert("RGB")

print("Base image size:", base.size)
print("Base pixel range:", np.array(base).min(), np.array(base).max())

print("LoRA image size:", lora.size)
print("LoRA pixel range:", np.array(lora).min(), np.array(lora).max())

display(base)
display(lora)

In [ ]:
prompt = "a drawing of a cartoon character holding a wrench"

lora_image = pipe(
    prompt,
    num_inference_steps=30,
    guidance_scale=7.5
).images[0]

display(lora_image)

In [ ]:
lora_image.save("task1_lora_image.png")
print("LoRA image saved successfully!")

In [ ]:
from PIL import Image, ImageDraw

# Make sure both images are RGB
base = base_image.convert("RGB")
lora = lora_image.convert("RGB")

# Resize LoRA image if necessary
lora = lora.resize(base.size)

# Create side-by-side image
comparison = Image.new(
    "RGB",
    (base.width * 2, base.height),
    "white"
)

comparison.paste(base, (0, 0))
comparison.paste(lora, (base.width, 0))

# Add labels
draw = ImageDraw.Draw(comparison)
draw.text((20, 20), "BASE MODEL", fill="black")
draw.text((base.width + 20, 20), "LO-RA REFINED", fill="black")

display(comparison)

# Save comparison
comparison.save("task1_base_vs_lora_comparison.png")

print("Comparison saved successfully!")

In [ ]:
import shutil
import os

# Create a folder for the final Task 1 results
os.makedirs("Task_1_Final", exist_ok=True)

# Copy important files
shutil.copy(
    "task1_lora_output/pytorch_lora_weights.safetensors",
    "Task_1_Final/pytorch_lora_weights.safetensors"
)

shutil.copy(
    "task1_base_image.png",
    "Task_1_Final/task1_base_image.png"
)

shutil.copy(
    "task1_lora_image.png",
    "Task_1_Final/task1_lora_image.png"
)

shutil.copy(
    "task1_base_vs_lora_comparison.png",
    "Task_1_Final/task1_base_vs_lora_comparison.png"
)

# Create ZIP
shutil.make_archive(
    "Task_1_Final",
    "zip",
    "Task_1_Final"
)

print("Task 1 ZIP created successfully!")
print("File: Task_1_Final.zip")

In [ ]:
from google.colab import files

files.download("Task_1_Final.zip")

In [ ]:
readme_content = """# Task 1 – Pretrained Text-to-Image Model Refinement Using LoRA

## Objective

The objective of this task was to refine a pretrained text-to-image model using a custom image-caption dataset. Stable Diffusion 1.5 was used as the pretrained model, with LoRA for parameter-efficient fine-tuning.

## Dataset

The custom dataset used was the Digimon BLIP Captions dataset from Hugging Face.

- Image-text pairs: 1,071
- Image format: RGB
- Training resolution: 512 x 512
- Captions: Text descriptions corresponding to the images

## Preprocessing

The images were converted to RGB format and resized to 512 x 512 pixels.

A metadata.jsonl file was created to associate each image with its corresponding caption.

The final dataset was verified using the Hugging Face imagefolder loader with the columns image and text.

## Pretrained Model

Model: runwayml/stable-diffusion-v1-5

The pretrained model was loaded using the Diffusers library and tested before training.

## LoRA Training

LoRA was used for parameter-efficient fine-tuning.

Training configuration:

- Base model: Stable Diffusion 1.5
- Dataset size: 1,071 images
- Resolution: 512 x 512
- Batch size: 1
- Gradient accumulation: 4
- Epochs: 1
- Learning rate: 0.0001
- LoRA rank: 4
- Precision: FP16
- GPU: NVIDIA Tesla T4
- Optimization steps: 268

## Training Result

Training completed successfully for one epoch.

Final reported step loss: 0.165

The trained LoRA weights were saved as:

task1_lora_output/pytorch_lora_weights.safetensors

## Image Generation

The trained LoRA adapter was successfully loaded into Stable Diffusion 1.5 and used to generate an image.

Example prompt:

"a drawing of a cartoon character holding a wrench"

## Base Model vs LoRA

The same prompt was used with the base model and the trained LoRA model.

The following files were created:

- task1_base_image.png
- task1_lora_image.png
- task1_base_vs_lora_comparison.png

The comparison provides a visual demonstration of the outputs before and after applying the trained LoRA adapter.

## Limitations

This was a proof-of-concept experiment using one training epoch and a relatively small custom dataset. Therefore, the results should not be interpreted as a comprehensive measurement of model improvement.

More training epochs, additional prompts, and quantitative or human evaluation could be used for a more detailed assessment.

## Conclusion

The task successfully demonstrated the process of refining a pretrained text-to-image model using a custom image-caption dataset and LoRA.

The trained LoRA adapter was successfully saved, loaded, and used to generate an image with Stable Diffusion 1.5.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("README.md created successfully!")

In [ ]:
with open("README.md", "r", encoding="utf-8") as f:
    content = f.read()

print(content[:2000])

In [ ]:
import shutil
import os

# Copy README into the final Task 1 folder
shutil.copy(
    "README.md",
    "Task_1_Final/README.md"
)

# Re-create the ZIP with the README included
shutil.make_archive(
    "Task_1_Final",
    "zip",
    "Task_1_Final"
)

print("Final Task 1 ZIP updated successfully!")
print("Contents:")

for file in os.listdir("Task_1_Final"):
    print("-", file)

In [ ]:
from google.colab import files

files.download("Task_1_Final.zip")